# Clasificación de Iris con Random Forest y MLflow

Este notebook entrena un clasificador, evalúa su desempeño y registra cada ejecución en MLflow. Está organizado para que la configuración, los datos, el entrenamiento y el seguimiento sean fáciles de reutilizar.

## Cómo pensar en MLflow

MLflow guarda de forma ordenada qué modelo se entrenó, con qué configuración, qué resultados obtuvo y qué archivos produjo.

- **Tracking URI**: indica dónde se guardan los datos de seguimiento. En Databricks normalmente se usa el entorno integrado; en local puede ser una carpeta como `./mlruns`.
- **Experimento**: agrupa entrenamientos relacionados. No es el modelo; es el contenedor lógico de varios intentos.
- **Run**: una ejecución concreta del entrenamiento. Cada run tiene un `run_id` único.
- **Parámetros**: decisiones usadas antes o durante el entrenamiento, como `n_estimators` o `max_depth`.
- **Métricas**: resultados numéricos, como accuracy y F1.
- **Tags**: etiquetas de contexto, como el tipo de modelo o la versión del proyecto.
- **Artefactos**: archivos producidos por la ejecución, como el modelo, un reporte JSON o una matriz de confusión.

La URI de seguimiento y la ubicación de artefactos son conceptos distintos: la primera conecta con el sistema de MLflow y la segunda indica dónde se guardan archivos grandes. Registrar un modelo no lo convierte automáticamente en producción; solo lo deja disponible para cargarlo o promoverlo después.

In [ ]:
import json
import logging
import os
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

## 1. Configuración

Todos los valores que normalmente cambiarían entre entornos están aquí. La idea es que una persona pueda adaptar el notebook cambiando esta sección, sin buscar rutas ni nombres repartidos por todo el código. En Databricks se pueden definir como variables de entorno antes de ejecutar el notebook. Si no se define `IRIS_DATA_PATH`, se usa `Iris.csv` en la carpeta actual.

Variables disponibles: `IRIS_DATA_PATH`, `MLFLOW_TRACKING_URI`, `IRIS_EXPERIMENT_NAME`, `IRIS_ARTIFACT_LOCATION`, `IRIS_MODEL_NAME`, `IRIS_DATASET_VERSION`, `IRIS_PROJECT_VERSION`, `IRIS_AUTHOR` e `IRIS_PURPOSE`.

In [ ]:
# Ruta del archivo de entrada. Aquí vive el dataset que se leerá con pandas.
# Puede ser una ruta local (por ejemplo, 'Iris.csv'), una ruta montada en
# Databricks o una ruta absoluta. Esta ruta solo se usa para LEER los datos;
# no se escribirá ni se modificará el archivo original.
# Se recomienda configurarla con IRIS_DATA_PATH para no cambiar el notebook
# cuando el proyecto se ejecute en otra máquina o entorno.
IS_DATABRICKS = ("dbutils" in globals()) or bool(os.getenv("DATABRICKS_RUNTIME_VERSION"))


def get_setting(env_name: str, default: str, widget_label: str) -> str:
    """Obtiene una configuración desde entorno, widget de Databricks o default."""
    env_value = os.getenv(env_name, "").strip()
    if env_value:
        return env_value
    if IS_DATABRICKS:
        try:
            dbutils.widgets.text(env_name, default, widget_label)
        except Exception:
            # El widget puede existir si el notebook ya se ejecutó.
            pass
        try:
            widget_value = dbutils.widgets.get(env_name).strip()
            return widget_value or default
        except Exception:
            pass
    return default


DEFAULT_DATA_PATH = (
    "/Volumes/workspace/my_data/my_volumen/Iris.csv"
    if IS_DATABRICKS
    else "Iris.csv"
)
DATA_PATH = Path(get_setting("IRIS_DATA_PATH", DEFAULT_DATA_PATH, "Ruta del dataset"))

# Dirección del servicio de MLflow que recibirá experimentos y ejecuciones.
# Esta variable NO es la ruta del dataset ni la carpeta donde se guardará
# automáticamente el modelo. Ejemplos: 'databricks', 'http://servidor:5000'
# o una configuración local. Si queda vacía, MLflow conserva su configuración
# predeterminada del entorno.
TRACKING_URI = get_setting("MLFLOW_TRACKING_URI", "databricks" if IS_DATABRICKS else "", "Tracking URI de MLflow")

# Nombre lógico del experimento. Un experimento agrupa múltiples runs o
# entrenamientos relacionados y permite compararlos en la interfaz de MLflow.
# No debe confundirse con el nombre del modelo: el experimento es el
# contenedor de ejecuciones, no el objeto que después se utiliza para predecir.
EXPERIMENT_NAME = get_setting("IRIS_EXPERIMENT_NAME", "/Shared/iris_mlflow" if IS_DATABRICKS else "iris_mlflow", "Experimento MLflow")

# Ubicación donde MLflow guardará artefactos grandes, como el modelo, el
# reporte JSON y la matriz de confusión. Es distinta de TRACKING_URI: la
# tracking URI conecta con MLflow; esta ubicación almacena archivos.
# Puede ser una ruta de DBFS/volumen en Databricks o una ubicación soportada
# por MLflow. Si queda vacía, se utiliza la ubicación configurada por MLflow.
ARTIFACT_LOCATION = get_setting("IRIS_ARTIFACT_LOCATION", "", "Ubicación de artefactos (opcional)")

# Nombre descriptivo de la ejecución. Ayuda a reconocer el entrenamiento en
# MLflow, pero no cambia el algoritmo ni crea por sí solo un modelo en producción.
MODEL_NAME = get_setting("IRIS_MODEL_NAME", "iris-random-forest", "Nombre del run")

# Identificador humano de la versión u origen del dataset. MLflow no calcula
# automáticamente si el CSV cambió; por eso este valor debe actualizarse
# cuando se reemplace, limpie o regenere el archivo de datos.
DATASET_VERSION = get_setting("IRIS_DATASET_VERSION", "iris-csv", "Versión del dataset")

# Versión del código o del proyecto. Sirve para distinguir resultados
# producidos por distintas versiones del notebook.
PROJECT_VERSION = get_setting("IRIS_PROJECT_VERSION", "1.0.0", "Versión del proyecto")

# Responsable y finalidad de la ejecución. Son metadatos que se guardan
# como tags en MLflow para facilitar búsquedas y auditoría. No afectan
# las predicciones del modelo.
AUTHOR = get_setting("IRIS_AUTHOR", "unknown", "Responsable")
PURPOSE = get_setting("IRIS_PURPOSE", "baseline-classification", "Propósito")

# Porción del dataset que se reserva para medir el modelo con datos que no
# utilizó durante el entrenamiento. 0.20 significa 20% para prueba y 80%
# para entrenamiento.
TEST_SIZE = 0.20

# Semilla que hace reproducible la división de datos y el entrenamiento.
# Mantenerla fija permite comparar runs; cambiarla permite comprobar la
# sensibilidad del resultado a otra división aleatoria.
RANDOM_STATE = 42

# Cantidad de árboles del Random Forest. Más árboles pueden estabilizar el
# resultado, pero también aumentan tiempo y consumo de recursos.
N_ESTIMATORS = 100

# Profundidad máxima de cada árbol. Un valor bajo limita la complejidad;
# un valor alto puede aprender demasiado los datos de entrenamiento.
MAX_DEPTH = 4

# Métrica usada como referencia principal al comparar runs. Debe ser una
# métrica registrada con el mismo nombre en all_metrics.
PRIMARY_METRIC = "test_f1_weighted"

if not 0 < TEST_SIZE < 1:
    raise ValueError("TEST_SIZE debe estar entre 0 y 1.")
if PRIMARY_METRIC not in {"test_accuracy", "test_f1_weighted"}:
    raise ValueError("PRIMARY_METRIC no está soportada.")

if TRACKING_URI:
    mlflow.set_tracking_uri(TRACKING_URI)

print(f"Dataset: {DATA_PATH}")
print(f"Experimento: {EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 2. Carga y validación de datos

La validación falla temprano y con un mensaje concreto. Esto evita entrenar un modelo con columnas equivocadas o datos incompletos.

In [ ]:
TARGET_COLUMN = "Species"
ID_COLUMN = "Id"


def load_and_validate_data(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """Carga Iris.csv y devuelve el dataframe, X e y validados."""
    if not path.is_file():
        raise FileNotFoundError(
            f"No existe el dataset en '{path}'. Define IRIS_DATA_PATH con la ruta correcta."
        )

    dataframe = pd.read_csv(path)
    required_columns = {ID_COLUMN, TARGET_COLUMN}
    missing_columns = required_columns - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"Faltan columnas obligatorias: {sorted(missing_columns)}.")

    if dataframe.empty:
        raise ValueError("El dataset está vacío.")
    if dataframe.isna().any().any():
        null_columns = dataframe.columns[dataframe.isna().any()].tolist()
        raise ValueError(f"Hay valores nulos en: {null_columns}.")
    if dataframe[TARGET_COLUMN].nunique() < 2:
        raise ValueError("El objetivo debe contener al menos dos clases.")

    feature_columns = [column for column in dataframe.columns if column not in {ID_COLUMN, TARGET_COLUMN}]
    if not feature_columns:
        raise ValueError("No se encontraron columnas predictoras.")
    non_numeric = dataframe[feature_columns].select_dtypes(exclude=np.number).columns.tolist()
    if non_numeric:
        raise TypeError(f"Las columnas predictoras deben ser numéricas: {non_numeric}.")

    X = dataframe[feature_columns].copy()
    y = dataframe[TARGET_COLUMN].copy().astype(str)
    return dataframe, X, y


df, X, y = load_and_validate_data(DATA_PATH)
print(f"Registros: {len(df)} | Variables: {list(X.columns)} | Clases: {sorted(y.unique())}")
display(df.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)


def create_model() -> RandomForestClassifier:
    """Crea el modelo usando únicamente la configuración central."""
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        random_state=RANDOM_STATE,
    )


def evaluate_model(model, features, target) -> tuple[dict, dict, np.ndarray]:
    """Calcula métricas agregadas, reporte por clase y matriz de confusión."""
    predictions = model.predict(features)
    precision, recall, f1, _ = precision_recall_fscore_support(
        target, predictions, average="weighted", zero_division=0
    )
    metrics = {
        "accuracy": float(accuracy_score(target, predictions)),
        "precision_weighted": float(precision),
        "recall_weighted": float(recall),
        "f1_weighted": float(f1),
    }
    report = classification_report(target, predictions, output_dict=True, zero_division=0)
    matrix = confusion_matrix(target, predictions, labels=sorted(y.unique()))
    return metrics, report, matrix


model = create_model()
model.fit(X_train, y_train)
train_metrics, train_report, _ = evaluate_model(model, X_train, y_train)
test_metrics, test_report, test_matrix = evaluate_model(model, X_test, y_test)

all_metrics = {
    **{f"train_{name}": value for name, value in train_metrics.items()},
    **{f"test_{name}": value for name, value in test_metrics.items()},
}
print(json.dumps(all_metrics, indent=2))

## 3. Registro completo en MLflow

Dentro de un `run` se guardan parámetros, métricas, tags, reportes y el modelo. Así se puede responder qué configuración produjo un resultado y recuperar el modelo posteriormente.

La firma (`signature`) describe las columnas y tipos de entrada que espera el modelo. El `input_example` muestra una entrada pequeña válida. Ambos ayudan a detectar errores cuando el modelo se reutiliza. El modelo se guarda como artefacto en `modelo`; esto no significa que esté desplegado en producción.

In [ ]:
def get_or_create_experiment(name: str, artifact_location: str = "") -> str:
    """Reutiliza el experimento si existe y lo crea solo una vez."""
    client = mlflow.MlflowClient()
    existing = client.get_experiment_by_name(name)
    if existing is not None:
        return existing.experiment_id
    if artifact_location:
        return client.create_experiment(name=name, artifact_location=artifact_location)
    return client.create_experiment(name=name)


experiment_id = get_or_create_experiment(EXPERIMENT_NAME, ARTIFACT_LOCATION)
mlflow.set_experiment(EXPERIMENT_NAME)

model_parameters = {
    "n_estimators": N_ESTIMATORS,
    "max_depth": MAX_DEPTH,
    "random_state": RANDOM_STATE,
}
run_parameters = {
    **model_parameters,
    "test_size": TEST_SIZE,
    "dataset_version": DATASET_VERSION,
    "feature_columns": json.dumps(list(X.columns)),
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "class_count": y.nunique(),
}
run_tags = {
    "model_type": "RandomForestClassifier",
    "dataset": DATASET_VERSION,
    "project_version": PROJECT_VERSION,
    "author": AUTHOR,
    "purpose": PURPOSE,
}

with mlflow.start_run(run_name=MODEL_NAME, experiment_id=experiment_id) as run:
    mlflow.log_params(run_parameters)
    mlflow.set_tags(run_tags)
    mlflow.log_metrics(all_metrics)
    mlflow.set_tag("primary_metric", PRIMARY_METRIC)

    mlflow.log_dict(test_report, "classification_report.json")

    figure, axis = plt.subplots(figsize=(6, 5))
    axis.imshow(test_matrix, cmap="Blues")
    axis.set(
        xticks=range(len(sorted(y.unique()))),
        yticks=range(len(sorted(y.unique()))),
        xticklabels=sorted(y.unique()),
        yticklabels=sorted(y.unique()),
        xlabel="Predicción",
        ylabel="Valor real",
        title="Matriz de confusión",
    )
    for row in range(test_matrix.shape[0]):
        for column in range(test_matrix.shape[1]):
            axis.text(column, row, test_matrix[row, column], ha="center", va="center")
    figure.tight_layout()
    # El driver de Databricks puede no permitir escribir en la carpeta del
    # notebook. Se usa una carpeta temporal local y solo se sube el archivo
    # terminado a MLflow.
    temporary_dir = tempfile.TemporaryDirectory()
    confusion_matrix_path = Path(temporary_dir.name) / "confusion_matrix.png"
    figure.savefig(confusion_matrix_path, dpi=150)
    plt.close(figure)
    mlflow.log_artifact(str(confusion_matrix_path))
    temporary_dir.cleanup()

    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="modelo",
        signature=signature,
        input_example=X_train.head(5),
    )

    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/modelo"
    artifact_uri = mlflow.get_artifact_uri()

print(f"Experimento: {EXPERIMENT_NAME} ({experiment_id})")
print(f"Run ID: {run_id}")
print(f"Métrica principal ({PRIMARY_METRIC}): {all_metrics[PRIMARY_METRIC]:.4f}")
print(f"Modelo MLflow: {model_uri}")
print(f"Artefactos: {artifact_uri}")

## 4. Verificación del modelo registrado

La siguiente comprobación carga el modelo desde MLflow y verifica que puede producir predicciones. El `run_id` permite volver a consultar exactamente esta ejecución. Si se vuelve a ejecutar el notebook, se crea otro run; eso es esperado y permite comparar configuraciones.

In [ ]:
loaded_model = mlflow.sklearn.load_model(model_uri)
loaded_predictions = loaded_model.predict(X_test.head(3))
print(f"Modelo cargado correctamente. Predicciones de ejemplo: {loaded_predictions.tolist()}")
print("Para comparar runs, abre el experimento en MLflow y compara parámetros, métricas y artefactos.")